# CICIDS2017 and CICIDS2018 Data Processing

This notebook processes the CICIDS2017 and CICIDS2018 datasets for the ThreatVision_AI project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib

# Set paths for datasets
cicids2017_path = '../data/MachineLearningCSV (2017)/MachineLearningCVE'
cicids2018_path = '../data/MachineLearningCSV (2018)/MachineLearningCVE'

# Output path for processed data
output_path = '../data/'

## Load and Explore CICIDS2017 Dataset

In [ ]:
# Get all CSV files in the 2017 dataset
cicids2017_files = glob.glob(os.path.join(cicids2017_path, '*.csv'))
print(f"Found {len(cicids2017_files)} files in CICIDS2017 dataset:")
for file in cicids2017_files:
    print(f"- {os.path.basename(file)}")

In [ ]:
# Load a sample file to explore the structure
sample_file_2017 = cicids2017_files[0]
print(f"Loading sample file: {os.path.basename(sample_file_2017)}")

# Read with error handling for potential encoding issues
try:
    df_2017_sample = pd.read_csv(sample_file_2017)
except UnicodeDecodeError:
    df_2017_sample = pd.read_csv(sample_file_2017, encoding='latin1')

print(f"Shape: {df_2017_sample.shape}")
print(f"Columns: {len(df_2017_sample.columns)}")
df_2017_sample.head()

## Load and Explore CICIDS2018 Dataset

In [ ]:
# Get all CSV files in the 2018 dataset
cicids2018_files = glob.glob(os.path.join(cicids2018_path, '*.csv'))
print(f"Found {len(cicids2018_files)} files in CICIDS2018 dataset:")
for file in cicids2018_files:
    print(f"- {os.path.basename(file)}")

In [ ]:
# Load a sample file to explore the structure
sample_file_2018 = cicids2018_files[0]
print(f"Loading sample file: {os.path.basename(sample_file_2018)}")

# Read with error handling for potential encoding issues
try:
    df_2018_sample = pd.read_csv(sample_file_2018)
except UnicodeDecodeError:
    df_2018_sample = pd.read_csv(sample_file_2018, encoding='latin1')

print(f"Shape: {df_2018_sample.shape}")
print(f"Columns: {len(df_2018_sample.columns)}")
df_2018_sample.head()

## Compare Dataset Structures

In [ ]:
# Compare columns between datasets
cols_2017 = set(df_2017_sample.columns)
cols_2018 = set(df_2018_sample.columns)

print(f"CICIDS2017 columns: {len(cols_2017)}")
print(f"CICIDS2018 columns: {len(cols_2018)}")
print(f"Common columns: {len(cols_2017.intersection(cols_2018))}")
print(f"Columns only in 2017: {len(cols_2017 - cols_2018)}")
print(f"Columns only in 2018: {len(cols_2018 - cols_2017)}")

## Process and Combine Datasets

In [ ]:
def load_and_process_dataset(file_paths, year):
    """Load and process multiple CSV files from a dataset."""
    all_data = []
    
    for file_path in file_paths:
        print(f"Processing {os.path.basename(file_path)}...")
        try:
            df = pd.read_csv(file_path)
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='latin1')
            
        # Clean column names (remove leading/trailing spaces)
        df.columns = [col.strip() for col in df.columns]
        
        # Add dataset year as a feature
        df['Dataset_Year'] = year
        
        # Add attack type based on filename
        filename = os.path.basename(file_path).lower()
        if 'ddos' in filename:
            df['Attack_Type'] = 'DDoS'
        elif 'portscan' in filename:
            df['Attack_Type'] = 'PortScan'
        elif 'infilteration' in filename or 'infiltration' in filename:
            df['Attack_Type'] = 'Infiltration'
        elif 'webattacks' in filename:
            df['Attack_Type'] = 'WebAttack'
        else:
            df['Attack_Type'] = 'Normal'
            
        all_data.append(df)
        
    # Combine all dataframes
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f"Combined shape: {combined_df.shape}")
    
    return combined_df

In [ ]:
# Process datasets (use a sample of files to avoid memory issues)
# Adjust the number of files based on your system's memory capacity
cicids2017_sample_files = cicids2017_files[:2]  # Start with just 2 files
cicids2018_sample_files = cicids2018_files[:2]  # Start with just 2 files

df_2017 = load_and_process_dataset(cicids2017_sample_files, 2017)
df_2018 = load_and_process_dataset(cicids2018_sample_files, 2018)

## Standardize Label Column

In [ ]:
# Check label column names and values
label_col_2017 = 'Label' if 'Label' in df_2017.columns else ' Label'
label_col_2018 = 'Label' if 'Label' in df_2018.columns else ' Label'

print("CICIDS2017 unique labels:")
print(df_2017[label_col_2017].value_counts())

print("\nCICIDS2018 unique labels:")
print(df_2018[label_col_2018].value_counts())

In [ ]:
# Standardize labels to binary classification (normal vs attack)
def standardize_labels(df, label_col):
    df['standardized_label'] = df[label_col].apply(lambda x: 0 if str(x).lower() == 'benign' or str(x).lower() == 'normal' else 1)
    return df

df_2017 = standardize_labels(df_2017, label_col_2017)
df_2018 = standardize_labels(df_2018, label_col_2018)

# Check standardized labels
print("CICIDS2017 standardized labels:")
print(df_2017['standardized_label'].value_counts())

print("\nCICIDS2018 standardized labels:")
print(df_2018['standardized_label'].value_counts())

## Combine Datasets with Common Features

In [ ]:
# Find common numeric columns between datasets
common_cols = list(set(df_2017.columns).intersection(set(df_2018.columns)))
print(f"Number of common columns: {len(common_cols)}")

# Filter to keep only numeric columns and the standardized label
numeric_cols = []
for col in common_cols:
    if df_2017[col].dtype in ['int64', 'float64'] and df_2018[col].dtype in ['int64', 'float64']:
        numeric_cols.append(col)

print(f"Number of common numeric columns: {len(numeric_cols)}")

# Create combined dataset with common numeric features
features_to_use = numeric_cols + ['standardized_label', 'Dataset_Year', 'Attack_Type']
df_2017_subset = df_2017[features_to_use]
df_2018_subset = df_2018[features_to_use]

combined_df = pd.concat([df_2017_subset, df_2018_subset], ignore_index=True)
print(f"Combined dataset shape: {combined_df.shape}")

## Data Visualization

In [ ]:
# Distribution of target variable
plt.figure(figsize=(10, 6))
sns.countplot(x='standardized_label', data=combined_df)
plt.title('Distribution of Target Variable (0=Normal, 1=Attack)')
plt.xlabel('Label')
plt.show()

In [ ]:
# Distribution of attack types
plt.figure(figsize=(12, 6))
sns.countplot(x='Attack_Type', data=combined_df)
plt.title('Distribution of Attack Types')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Compare datasets
plt.figure(figsize=(10, 6))
sns.countplot(x='Dataset_Year', hue='standardized_label', data=combined_df)
plt.title('Attack Distribution by Dataset Year')
plt.xlabel('Dataset Year')
plt.ylabel('Count')
plt.legend(['Normal', 'Attack'])
plt.show()

## Data Preprocessing

In [ ]:
# Handle missing values
combined_df = combined_df.fillna(0)

# Select features and target
X = combined_df.drop(['standardized_label', 'Dataset_Year', 'Attack_Type'], axis=1)
y = combined_df['standardized_label']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA for dimensionality reduction
pca = PCA(n_components=20)  # Adjust number of components as needed
X_pca = pca.fit_transform(X_scaled)

print(f"Original feature count: {X.shape[1]}")
print(f"PCA feature count: {X_pca.shape[1]}")
print(f"Explained variance ratio: {sum(pca.explained_variance_ratio_):.4f}")

## Save Processed Data and Models

In [ ]:
# Create a DataFrame with PCA features
pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
pca_df['label'] = y.values
pca_df['dataset_year'] = combined_df['Dataset_Year'].values
pca_df['attack_type'] = combined_df['Attack_Type'].values

# Save processed data
os.makedirs(output_path, exist_ok=True)
pca_df.to_csv(os.path.join(output_path, 'cicids_combined_data.csv'), index=False)
print(f"Saved processed data to {os.path.join(output_path, 'cicids_combined_data.csv')}")

# Save scaler and PCA models
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/threatvision_scaler.pkl')
joblib.dump(pca, '../models/threatvision_pca.pkl')
print("Saved scaler and PCA models")